# 01 - Data Version Control with Azure ML

**Azure ML MLOps Workshop | Block 1**

### Why Data Versioning?

In production ML, data changes constantly. New inspections come in from the field,
labeling gets corrected, preprocessing logic evolves. Without versioning:
- You can't reproduce a model's results
- You can't roll back when new data degrades performance
- You can't audit which data trained which model

Azure ML **Data Assets** solve this by treating datasets as immutable, versioned artifacts.

### What you'll do:
1. Upload raw datasets to the workspace datastore
2. Register them as versioned Data Assets (v1 = raw)
3. Clean and preprocess the data
4. Register the cleaned version (v2 = cleaned)
5. Explore versioning and lineage

---

In [ ]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

SUBSCRIPTION_ID = "<YOUR_SUBSCRIPTION_ID>"  # <<<< CHANGE THIS TO YOUR AZURE SUBSCRIPTION ID
RESOURCE_GROUP = "<YOUR_RESOURCE_GROUP>"  # <<<< CHANGE THIS TO YOUR RESOURCE GROUP (e.g., rg-aml-workshop-jd)
WORKSPACE_NAME = "<YOUR_WORKSPACE_NAME>"  # <<<< CHANGE THIS TO YOUR WORKSPACE NAME (e.g., aml-workshop-jd)

ml_client = MLClient(
    credential=DefaultAzureCredential(),
    subscription_id=SUBSCRIPTION_ID,
    resource_group_name=RESOURCE_GROUP,
    workspace_name=WORKSPACE_NAME,
)
print(f"Connected to: {ml_client.workspace_name}")

## Step 1: Register Raw Data as Version 1

We register the CSV files directly as `uri_file` data assets. This uploads them
to the workspace's default blob storage and creates a versioned reference.

In [ ]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

inspections_data_v1 = Data(
    name="classified-inspections",
    version="1",
    description="Raw Contoso classified inspections dataset - inspection comments with lead opportunity labels.",
    path="../../data/inspections_dataset.csv",
    type=AssetTypes.URI_FILE,
    tags={
        "source": "contoso",
        "stage": "raw",
        "region": "region-1",
        "rows": "10500",
    },
)

inspections_data_v1 = ml_client.data.create_or_update(inspections_data_v1)
print(f"Registered: {inspections_data_v1.name} v{inspections_data_v1.version}")
print(f"  Path: {inspections_data_v1.path}")

In [ ]:
os_data_v1 = Data(
    name="service-orders",
    version="1",
    description="Raw Contoso service orders dataset - equipment maintenance records for heavy equipment.",
    path="../../data/service_orders_dataset.csv",
    type=AssetTypes.URI_FILE,
    tags={
        "source": "contoso",
        "stage": "raw",
        "region": "region-1",
        "rows": "425745",
    },
)

os_data_v1 = ml_client.data.create_or_update(os_data_v1)
print(f"Registered: {os_data_v1.name} v{os_data_v1.version}")
print(f"  Path: {os_data_v1.path}")

### Check in Azure ML Studio

Go to **Azure ML Studio > Data** in the left navigation.
You should see both data assets listed with version 1.

---

## Step 2: Profile the Raw Data

Before cleaning, let's understand what we're working with.

In [ ]:
import pandas as pd

df_insp = pd.read_csv("../../data/inspections_dataset.csv")

print(f"Shape: {df_insp.shape}")
print(f"\nNull counts:\n{df_insp.isnull().sum()}")
print(f"\nLabel distribution:")
print(df_insp["is_lead_opportunity"].value_counts())
print(f"\nConfidence distribution:")
print(df_insp["confidence"].value_counts())
print(f"\nAvg comment length: {df_insp['comment'].str.len().mean():.1f} chars")

In [ ]:
print("=== LEAD OPPORTUNITIES (is_lead_opportunity=True) ===")
for c in df_insp[df_insp["is_lead_opportunity"] == True]["comment"].head(5).values:
    print(f"  > {c}")

print("\n=== NOT LEAD (is_lead_opportunity=False) ===")
for c in df_insp[df_insp["is_lead_opportunity"] == False]["comment"].head(5).values:
    print(f"  > {c}")

## Step 3: Clean and Preprocess

We'll create a cleaned version that:
- Removes HTML tags (`<br/>` etc.)
- Normalizes whitespace and casing
- Adds a numeric `label` column
- Saves as CSV for more efficient downstream processing

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../../src/track_a_text"))

from preprocess import load_and_clean_inspections

df_clean = load_and_clean_inspections("../../data/inspections_dataset.csv")

print(f"Cleaned shape: {df_clean.shape}")
print(f"\nSample cleaned comments:")
for _, row in df_clean.head(5).iterrows():
    print(f"  [{row['label']}] {row['comment_clean'][:80]}")

In [ ]:
cleaned_path = "../../data/inspections_cleaned.csv"
df_clean.to_csv(cleaned_path, index=False, encoding="utf-8")
print(f"Saved cleaned data to {cleaned_path}")
print(f"  Rows: {len(df_clean):,}")

## Step 4: Register Cleaned Data as Version 2

This is the key MLOps pattern: every transformation of the data gets its own version.
The original raw data (v1) remains untouched and immutable.

In [ ]:
inspections_data_v2 = Data(
    name="classified-inspections",
    version="2",
    description="Cleaned Contoso inspections - HTML removed, text normalized, label column added. Ready for ML training.",
    path=cleaned_path,
    type=AssetTypes.URI_FILE,
    tags={
        "source": "contoso",
        "stage": "cleaned",
        "region": "region-1",
        "rows": str(len(df_clean)),
        "parent_version": "1",
        "preprocessing": "html_removal,lowercase,whitespace_norm",
    },
)

inspections_data_v2 = ml_client.data.create_or_update(inspections_data_v2)
print(f"Registered: {inspections_data_v2.name} v{inspections_data_v2.version}")
print(f"  Tags: {inspections_data_v2.tags}")

## Step 5: Explore Versioning & Lineage

Let's demonstrate how to query and navigate versions programmatically.

In [ ]:
print("All versions of 'classified-inspections':")
for version in ml_client.data.list(name="classified-inspections"):
    print(f"  v{version.version} | {version.description[:60]}... | Stage: {version.tags.get('stage', 'N/A')}")

In [ ]:
data_v1 = ml_client.data.get(name="classified-inspections", version="1")
data_v2 = ml_client.data.get(name="classified-inspections", version="2")

print(f"v1 path: {data_v1.path}")
print(f"v2 path: {data_v2.path}")
print(f"\nv2 was derived from v1 (tag: parent_version={data_v2.tags.get('parent_version')})")

In [ ]:
latest = ml_client.data.get(name="classified-inspections", label="latest")
print(f"Latest version: v{latest.version}")

## Key Takeaways

| Concept | What We Did |
|---------|------------|
| **Immutability** | v1 (raw) can never be modified once registered |
| **Lineage** | v2 tags reference v1, documenting the transformation chain |
| **Reproducibility** | Any experiment can reference a specific version |
| **Multi-region** | Tags include `region` - each region can register its own data assets or consume shared ones |

### Go to Azure ML Studio now

Navigate to **Data > classified-inspections** and click through both versions.
Notice how each version has its own metadata, tags, and storage path.

---

**Next:** [02 - Experiment Tracking](./02_experiment_tracking.ipynb) | [01b - Data Versioning (Tabular)](../track_b_tabular/01b_data_versioning_tabular.ipynb)